In [68]:
from sqlalchemy import create_engine
from dotenv import load_dotenv
from urllib.parse import quote_plus
import pandas as pd
import os
import re 



In [69]:
load_dotenv()

# quote_plus special characters ko encode karta hai
password = quote_plus(os.getenv('DB_PASSWORD'))

engine = create_engine(
    f"postgresql+psycopg2://{os.getenv('DB_USER')}:{password}@{os.getenv('DB_HOST')}/{os.getenv('DB_NAME')}"
)

df = pd.read_sql("SELECT * FROM transformed_jobs", engine)
print(df.shape)
print(f"Loaded {len(df)} clean rows")
print(f"\nNull values:\n{df.isnull().sum()}")
print(f"\nData types:\n{df.dtypes}")

(64, 11)
Loaded 64 clean rows

Null values:
id                     0
job_title              0
company_name           0
job_description        0
job_employment_type    0
job_city               0
job_country            0
job_apply_link         0
posted_date            0
created_at             0
role_searched          0
dtype: int64

Data types:
id                              int64
job_title                      object
company_name                   object
job_description                object
job_employment_type            object
job_city                       object
job_country                    object
job_apply_link                 object
posted_date            datetime64[ns]
created_at             datetime64[ns]
role_searched                  object
dtype: object


In [70]:
df.shape

(64, 11)

In [71]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 64 entries, 0 to 63
Data columns (total 11 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   id                   64 non-null     int64         
 1   job_title            64 non-null     object        
 2   company_name         64 non-null     object        
 3   job_description      64 non-null     object        
 4   job_employment_type  64 non-null     object        
 5   job_city             64 non-null     object        
 6   job_country          64 non-null     object        
 7   job_apply_link       64 non-null     object        
 8   posted_date          64 non-null     datetime64[ns]
 9   created_at           64 non-null     datetime64[ns]
 10  role_searched        64 non-null     object        
dtypes: datetime64[ns](2), int64(1), object(8)
memory usage: 5.6+ KB


In [72]:
df.describe()

,id,posted_date,created_at
count,64.000000,64,64
mean,283.687500,2026-02-25 05:46:52.500000,2026-02-28 21:52:15.957296384
min,1.000000,2026-02-22 00:00:00,2026-02-28 16:33:14.406315
25%,306.750000,2026-02-23 18:00:00,2026-02-28 16:53:46.401641984
50%,323.500000,2026-02-26 00:00:00,2026-02-28 16:53:46.401641984
75%,351.250000,2026-02-27 00:00:00,2026-02-28 16:53:46.401641984
max,377.000000,2026-03-02 10:00:00,2026-03-02 22:31:58.720018
std,122.470972,NaN,NaN


In [73]:
df.isnull().sum()

id                     0
job_title              0
company_name           0
job_description        0
job_employment_type    0
job_city               0
job_country            0
job_apply_link         0
posted_date            0
created_at             0
role_searched          0
dtype: int64

In [74]:
df.duplicated().sum()


np.int64(0)

In [75]:
df.dtypes

id                              int64
job_title                      object
company_name                   object
job_description                object
job_employment_type            object
job_city                       object
job_country                    object
job_apply_link                 object
posted_date            datetime64[ns]
created_at             datetime64[ns]
role_searched                  object
dtype: object

In [76]:
df.isnull()

,id,job_title,company_name,job_description,job_employment_type,job_city,job_country,job_apply_link,posted_date,created_at,role_searched
0,False,False,False,False,False,False,False,False,False,False,False
1,False,False,False,False,False,False,False,False,False,False,False
2,False,False,False,False,False,False,False,False,False,False,False
3,False,False,False,False,False,False,False,False,False,False,False
4,False,False,False,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...
59,False,False,False,False,False,False,False,False,False,False,False
60,False,False,False,False,False,False,False,False,False,False,False
61,False,False,False,False,False,False,False,False,False,False,False
62,False,False,False,False,False,False,False,False,False,False,False


# ─── EXPLORATION SUMMARY ───────────────────────
# Shape: 381 rows, 11 columns
# Nulls: job_city (75), job_country (1)
# Duplicates: 0
# Dtypes: all object — needs conversion
# Issue: no salary column from free API tier
# ───────────────────────────────────────────────

In [77]:
print(df[["job_city"]].value_counts().head(3))
print(df["role_searched"].value_counts().head(3))

job_city 
Unknown      15
Bengaluru     7
Gurugram      7
Name: count, dtype: int64
role_searched
Data Scientist    36
Data Analyst      22
Data Engineer      6
Name: count, dtype: int64


In [78]:
df['job_employment_type']=df['job_employment_type'].str.strip().str.title()
print(df["job_employment_type"].value_counts())

job_employment_type
Full-Time     63
Contractor     1
Name: count, dtype: int64


In [79]:
df["job_employment_type"].head(3)

0    Full-Time
1    Full-Time
2    Full-Time
Name: job_employment_type, dtype: object

In [80]:
df.duplicated().sum()

np.int64(0)

In [81]:
# job_title + employer_name combination duplicates
df.duplicated(subset=["job_title", "job_city", "company_name"]).sum()


np.int64(0)

In [82]:
df.drop_duplicates(subset=["job_title", "job_city", "company_name"], keep='first', inplace=True) 
print(df.shape)

(64, 11)


In [83]:
print(f'final shape: {df.shape}')
print(f'final null values:\n{df.isnull().sum()}')
print(f'final data types:\n{df.dtypes}')


final shape: (64, 11)
final null values:
id                     0
job_title              0
company_name           0
job_description        0
job_employment_type    0
job_city               0
job_country            0
job_apply_link         0
posted_date            0
created_at             0
role_searched          0
dtype: int64
final data types:
id                              int64
job_title                      object
company_name                   object
job_description                object
job_employment_type            object
job_city                       object
job_country                    object
job_apply_link                 object
posted_date            datetime64[ns]
created_at             datetime64[ns]
role_searched                  object
dtype: object


In [84]:
df = pd.read_sql("SELECT * FROM raw_jobs", engine)

In [85]:
df["job_city"].fillna("Unknown", inplace=True)

C:\Users\aadit\AppData\Local\Temp\ipykernel_32676\4101281386.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df["job_city"].fillna("Unknown", inplace=True)


In [86]:
df["posted_date"]=pd.to_datetime(df["posted_date"], errors='coerce')
print(df.isnull().sum())
print(df['posted_date'].head(3))

id                     0
job_title              0
company_name           0
job_description        0
job_employment_type    0
job_city               0
job_country            1
job_apply_link         0
posted_date            0
created_at             0
role_searched          0
dtype: int64
0   2026-02-27 00:00:00
1   2026-02-27 00:00:00
2   2026-02-28 03:00:00
Name: posted_date, dtype: datetime64[ns]


In [87]:
df["job_city"]=df["job_city"].str.strip().str.title()
df["job_country"]=df["job_country"].str.strip().str.title()
df["job_title"]=df["job_title"].str.strip().str.title() 
df["role_searched"]=df["role_searched"].str.strip().str.title()
df["job_description"]=df["job_description"].str.strip()

In [88]:
df["job_country"].fillna("India", inplace=True)

C:\Users\aadit\AppData\Local\Temp\ipykernel_32676\2978140326.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df["job_country"].fillna("India", inplace=True)


In [89]:
df.columns

Index(['id', 'job_title', 'company_name', 'job_description',
       'job_employment_type', 'job_city', 'job_country', 'job_apply_link',
       'posted_date', 'created_at', 'role_searched'],
      dtype='object')

In [90]:
df.rename(columns={'employer_name': 'company_name'}, inplace=True)
print(df.columns.tolist())

['id', 'job_title', 'company_name', 'job_description', 'job_employment_type', 'job_city', 'job_country', 'job_apply_link', 'posted_date', 'created_at', 'role_searched']


In [91]:
df.duplicated(subset='job_apply_link').sum()

np.int64(317)

In [92]:
df['job_apply_link'].head(70)

0     https://in.linkedin.com/jobs/view/senior-data-...
1     https://jobs.comcast.com/job/chennai/data-anal...
2     https://www.efinancialcareers.com/jobs-India-B...
3     https://careers.mastercard.com/us/en/job/R-262...
4     https://www.shine.com/jobs/data-analyst-data-s...
                            ...                        
65    https://in.linkedin.com/jobs/view/senior-data-...
66    https://jobs.comcast.com/job/chennai/data-anal...
67    https://www.efinancialcareers.com/jobs-India-B...
68    https://careers.mastercard.com/us/en/job/R-262...
69    https://www.shine.com/jobs/data-analyst-data-s...
Name: job_apply_link, Length: 70, dtype: object

In [93]:
df.drop_duplicates(subset=["job_apply_link"], keep='first', inplace=True)

In [94]:
df.duplicated(['job_apply_link']).sum()

np.int64(0)

In [95]:
print(f"Final shape: {df.shape}")
print(f"Null values:\n{df.isnull().sum()}")
print(f"Duplicates: {df.duplicated().sum()}")

Final shape: (64, 11)
Null values:
id                     0
job_title              0
company_name           0
job_description        0
job_employment_type    0
job_city               0
job_country            0
job_apply_link         0
posted_date            0
created_at             0
role_searched          0
dtype: int64
Duplicates: 0


In [96]:
print(f"\nSample:\n{df.head(3)}")


Sample:
   id                                      job_title  \
0   1    Senior Data Analyst (Qa & Automation Focus)   
1   2                                 Data Analyst 3   
2   3  Senior Financial Data Analyst | Bangalore, In   

                   company_name  \
0  SID Information Technologies   
1                       Comcast   
2                       Moody's   

                                     job_description job_employment_type  \
0  Role: Senior Data Analyst (QA & Automation Foc...           Full-time   
1  Comcast brings together the best in media and ...           Full-time   
2  At Moody's, we unite the brightest minds to tu...           Full-time   

    job_city job_country                                     job_apply_link  \
0  Hyderabad          In  https://in.linkedin.com/jobs/view/senior-data-...   
1    Chennai          In  https://jobs.comcast.com/job/chennai/data-anal...   
2  Bengaluru          In  https://www.efinancialcareers.com/jobs-India-B...   

   

In [97]:
df.to_csv("cleaned_jobs.csv", index=False)
print("Cleaned data saved to cleaned_jobs.csv")

Cleaned data saved to cleaned_jobs.csv


In [98]:
df.to_sql(
    'transformed_jobs',  # table name
    engine,              # connection
    if_exists='replace', # agar table hai toh replace karo
    index=False          # DataFrame index mat daalo
)
print(f"✅ {len(df)} clean jobs loaded into transformed_jobs!")

✅ 64 clean jobs loaded into transformed_jobs!


In [ ]:
skills_list = [
    # Programming 
    'python', 'sql', 'java', 'scala',
    # DE Tools
    'airflow', 'spark', 'kafka', 'docker',
    'kubernetes', 'dbt',
    # Databases
    'postgresql', 'mysql', 'mongodb',
    'cassandra', 'redis', 'snowflake',
    # Cloud
    'aws', 'azure', 'gcp', 'redshift',
    'bigquery', 'databricks',
    # DA Tools
    'pandas', 'numpy', 'matplotlib',
    'seaborn', 'tableau', 'power bi',
    # ML
    'machine learning', 'deep learning',
    'tensorflow', 'pytorch', 'scikit-learn',
    # Others
    'git', 'linux', 'rest api', 'etl',
    'data pipeline', 'excel'
]

In [100]:
def extract_skills(description):
    if not description:
        return []
    
    description = description.lower()  # lowercase for matching
    found_skills = []
    
    for skill in skills_list:
        # Minimum 2 characters
        if len(skill) < 2:
            continue
        pattern = r'\b' + re.escape(skill) + r'\b'
        if re.search(pattern, description):
            found_skills.append(skill)
    
    return found_skills

In [101]:
sample=df["job_description"].iloc[0]
print(extract_skills(sample))

['python', 'sql', 'aws', 'tableau', 'etl']


In [106]:
df['skills_found'] = df['job_description'].apply(extract_skills)

all_skills = []
for skills in df['skills_found']:
    all_skills.extend(skills)

skill_counts = Counter(all_skills)
print(f"Average skills per job: {df['skills_found'].apply(len).mean():.1f}")


Average skills per job: 5.2


In [ ]:
job_skills_rows = []

for _, row in df.iterrows():
    for skill in row['skills_found']:
        job_skills_rows.append({
            'job_id': row['id'],
            'job_title': row['job_title'],
            'company_name': row['company_name'],
            'skill': skill,
            'role_searched': row['role_searched']
        })

job_skills_df = pd.DataFrame(job_skills_rows)
print(f"Total skill records: {len(job_skills_df)}")
print(job_skills_df.head(10))

Total skill records: 330
   job_id                                    job_title  \
0       1  Senior Data Analyst (Qa & Automation Focus)   
1       1  Senior Data Analyst (Qa & Automation Focus)   
2       1  Senior Data Analyst (Qa & Automation Focus)   
3       1  Senior Data Analyst (Qa & Automation Focus)   
4       1  Senior Data Analyst (Qa & Automation Focus)   
5       2                               Data Analyst 3   
6       2                               Data Analyst 3   
7       2                               Data Analyst 3   
8       2                               Data Analyst 3   
9       2                               Data Analyst 3   

                   company_name    skill   role_searched  
0  SID Information Technologies   python  Data Scientist  
1  SID Information Technologies      sql  Data Scientist  
2  SID Information Technologies      aws  Data Scientist  
3  SID Information Technologies  tableau  Data Scientist  
4  SID Information Technologies      etl 

In [104]:
job_skills_df.to_sql(
    'job_skills',
    engine,
    if_exists='replace',
    index=False
)
print(f"✅ {len(job_skills_df)} skill records loaded!")

✅ 330 skill records loaded!


In [105]:
print("Top 15 Most Demanded Skills:")
for skill, count in skill_counts.most_common(15):
    print(f"  {skill:20} → {count} jobs")

Top 15 Most Demanded Skills:
  python               → 38 jobs
  sql                  → 34 jobs
  machine learning     → 28 jobs
  azure                → 23 jobs
  power bi             → 23 jobs
  aws                  → 19 jobs
  tableau              → 18 jobs
  excel                → 14 jobs
  etl                  → 12 jobs
  pytorch              → 11 jobs
  tensorflow           → 11 jobs
  spark                → 10 jobs
  databricks           → 10 jobs
  scikit-learn         → 9 jobs
  kubernetes           → 8 jobs
